<a href="https://colab.research.google.com/github/dushyant-puro/8086-visual-cpu-simulator/blob/main/Spam_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# @title Default title text
# Spam Message Classifier

### Machine Learning Based SMS Spam Detection

This project classifies SMS messages as Spam or Not Spam using
Natural Language Processing and Machine Learning.

In [2]:
import zipfile
import os

zip_path = "/content/archive.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/spam_dataset")

print("Dataset extracted successfully!")

FileNotFoundError: [Errno 2] No such file or directory: '/content/archive.zip'

In [ ]:
import os

for root, dirs, files in os.walk("/content/spam_dataset"):
    print(root)
    for file in files:
        print("  ", file)

In [ ]:
import pandas as pd

df = pd.read_csv(
    "/content/spam_dataset/spam.csv",
    encoding="latin-1"
)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

df.head()

In [3]:
# Keep only the columns we need
df = df[['v1', 'v2']]

# Rename columns for clarity
df.columns = ['label', 'message']

# Convert labels to numerical values
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

# Check for missing values
print("Missing values:")
print(df.isnull().sum())

# Display the cleaned dataset
df.head()

NameError: name 'df' is not defined

In [ ]:
print("Number of messages:")
print(df.shape[0])

print("\nLabel distribution:")
print(df['label'].value_counts())

print("\nPercentage distribution:")
print(df['label'].value_counts(normalize=True) * 100)


In [4]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.countplot(x='label', data=df)

plt.xticks([0, 1], ['Not Spam', 'Spam'])
plt.title('Distribution of Spam and Not Spam Messages')
plt.xlabel('Message Type')
plt.ylabel('Number of Messages')
plt.show()

NameError: name 'df' is not defined

In [ ]:
import nltk

nltk.download('stopwords')

In [ ]:
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# English stop words
stop_words = set(stopwords.words('english'))

# Stemmer
stemmer = PorterStemmer()

def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()

    # Remove punctuation and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Split into individual words
    words = text.split()

    # Remove stop words and apply stemming
    words = [
        stemmer.stem(word)
        for word in words
        if word not in stop_words
    ]

    # Join words back into a sentence
    return ' '.join(words)

In [ ]:
df['clean_message'] = df['message'].apply(preprocess_text)

df[['message', 'clean_message']].head(10)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF vectorizer
tfidf = TfidfVectorizer()

# Convert cleaned messages into numerical features
X = tfidf.fit_transform(df['clean_message'])

print("TF-IDF vectorization completed!")
print("Number of messages:", X.shape[0])
print("Number of features:", X.shape[1])

In [ ]:
y = df['label']

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

In [ ]:
from sklearn.naive_bayes import MultinomialNB

# Create the Naive Bayes model
nb_model = MultinomialNB()

# Train the model
nb_model.fit(X_train, y_train)

# Make predictions on test data
nb_pred = nb_model.predict(X_test)

print("Naive Bayes model trained successfully!")

In [ ]:
from sklearn.linear_model import LogisticRegression

# Create the Logistic Regression model
lr_model = LogisticRegression(max_iter=1000)

# Train the model
lr_model.fit(X_train, y_train)

# Make predictions on test data
lr_pred = lr_model.predict(X_test)

print("Logistic Regression model trained successfully!")

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Calculate metrics for Naive Bayes
nb_accuracy = accuracy_score(y_test, nb_pred)
nb_precision = precision_score(y_test, nb_pred)
nb_recall = recall_score(y_test, nb_pred)
nb_f1 = f1_score(y_test, nb_pred)

# Calculate metrics for Logistic Regression
lr_accuracy = accuracy_score(y_test, lr_pred)
lr_precision = precision_score(y_test, lr_pred)
lr_recall = recall_score(y_test, lr_pred)
lr_f1 = f1_score(y_test, lr_pred)

# Display results
print("===== Naive Bayes =====")
print("Accuracy :", round(nb_accuracy, 4))
print("Precision:", round(nb_precision, 4))
print("Recall   :", round(nb_recall, 4))
print("F1 Score :", round(nb_f1, 4))

print("\n===== Logistic Regression =====")
print("Accuracy :", round(lr_accuracy, 4))
print("Precision:", round(lr_precision, 4))
print("Recall   :", round(lr_recall, 4))
print("F1 Score :", round(lr_f1, 4))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Naive Bayes confusion matrix
nb_cm = confusion_matrix(y_test, nb_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(
    nb_cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Not Spam', 'Spam'],
    yticklabels=['Not Spam', 'Spam']
)

plt.title('Naive Bayes - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


# Logistic Regression confusion matrix
lr_cm = confusion_matrix(y_test, lr_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(
    lr_cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Not Spam', 'Spam'],
    yticklabels=['Not Spam', 'Spam']
)

plt.title('Logistic Regression - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Naive Bayes', 'Logistic Regression'],
    'Accuracy': [nb_accuracy, lr_accuracy],
    'Precision': [nb_precision, lr_precision],
    'Recall': [nb_recall, lr_recall],
    'F1 Score': [nb_f1, lr_f1]
})

comparison.round(4)


In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Confusion Matrix - Naive Bayes
nb_cm = confusion_matrix(y_test, nb_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(
    nb_cm,
    annot=True,
    fmt='d',
    xticklabels=['Not Spam', 'Spam'],
    yticklabels=['Not Spam', 'Spam']
)

plt.title('Naive Bayes - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


# Confusion Matrix - Logistic Regression
lr_cm = confusion_matrix(y_test, lr_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(
    lr_cm,
    annot=True,
    fmt='d',
    xticklabels=['Not Spam', 'Spam'],
    yticklabels=['Not Spam', 'Spam']
)

plt.title('Logistic Regression - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


In [ ]:
comparison = pd.DataFrame({
    'Model': ['Naive Bayes', 'Logistic Regression'],
    'Accuracy': [nb_accuracy, lr_accuracy],
    'Precision': [nb_precision, lr_precision],
    'Recall': [nb_recall, lr_recall],
    'F1 Score': [nb_f1, lr_f1]
})

comparison.round(4)

In [ ]:
def predict_message(message):
    # Preprocess the new message
    cleaned_message = preprocess_text(message)

    # Convert the message into TF-IDF features
    message_tfidf = tfidf.transform([cleaned_message])

    # Predict using the better-performing model
    prediction = nb_model.predict(message_tfidf)[0]

    if prediction == 1:
        return "SPAM"
    else:
        return "NOT SPAM"

In [ ]:
test_messages = [
    "Congratulations! You have won a free prize. Call now!",
    "Hey, are we still meeting at 5 pm today?",
    "URGENT! You have won a cash reward. Claim now!",
    "Can you send me the notes from today's class?",
    "You have been selected for a free vacation. Call immediately!"
]

for message in test_messages:
    print("Message:", message)
    print("Prediction:", predict_message(message))
    print("-" * 70)

## Conclusion

In this project, an SMS Spam Classifier was developed using Natural Language Processing and Machine Learning.

The SMS messages were preprocessed by converting text to lowercase, removing punctuation and numbers, removing stop words, and applying stemming. TF-IDF was then used to convert the processed text into numerical features.

Two classification models were implemented: Multinomial Naive Bayes and Logistic Regression. The models were evaluated using Accuracy, Precision, Recall, and F1 Score, along with confusion matrices.

Based on the test results, Naive Bayes performed better than Logistic Regression, achieving an accuracy of 96.68% and an F1 Score of 85.82%. It also achieved 100% precision, meaning that the messages predicted as spam were correctly classified as spam in the test set.

The final system can also classify new, unseen SMS messages using the custom prediction function.